In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 4 — A PRIMEIRA LEI DA MÁQUINA: SEPARAÇÃO SAGRADA DOS DADOS

> "Aquele que conhece o futuro antes de vivê-lo nunca poderá prever o futuro verdadeiro. Ele apenas se lembrará do que já viu."
> — Inscrição em um antigo manual de modelagem preditiva

Três semanas após a carta de Helena, eu estava de volta à mesma mesa. Mas nada era igual. Cada comando carregava um peso novo — o peso da consciência de que código mal escrito não produz apenas *bugs*, produz consequências. O OncoClassify 1.0 era a prova. Ele era um modelo que sabia demais. Ele tinha espiado o futuro.

Meu erro fundamental, o pecado original do qual todos os outros derivaram, foi sutil. Antes de escrever a primeira linha do algoritmo, eu havia explorado o dataset **completo**. E, pior, eu havia deixado no modelo colunas que pareciam ótimos preditores sem perguntar *de onde* vinha aquele poder. Meu modelo não aprendia a prever o futuro: ele reconhecia um futuro que já tinha visto.

Desta vez, o primeiro ato seria o mais importante de todo o projeto: dividir os dados e criar um cofre. Dentro dele, o conjunto de teste permaneceria intocado — uma cápsula do tempo do mundo real. E, antes de confiar em qualquer *feature*, eu perguntaria de onde vinha o seu poder.

## A Regra de Ouro: Train-Test Split

Existe uma regra que não pode ser violada: **o conjunto de teste deve permanecer invisível até o momento final da avaliação.** Dividimos o dataset em dois subconjuntos independentes:

**Conjunto de Treino** (~80%) — usado para tudo: explorar, criar *features*, treinar, ajustar hiperparâmetros.

**Conjunto de Teste** (~20%) — trancado no "cofre", usado uma única vez, no fim, para simular dados que o modelo nunca viu.

O objetivo do *machine learning* não é acertar nos dados que já viu, mas **generalizar** para dados novos.

> **⚠️ ATENÇÃO — Data Leakage: o vazamento silencioso**
>
> *Data leakage* ocorre quando informação que não estaria disponível no momento real da previsão "vaza" para o treinamento. Leva a uma avaliação otimista e falsa: você pensa ter um ótimo modelo, e ele fracassa na produção. As formas mais comuns:
>
> **Explorar/pré-processar antes de separar** — estatísticas do teste influenciam o treino.
>
> **Selecionar *features* no dataset completo** — a informação do teste vaza na escolha.
>
> **Incluir colunas derivadas do próprio alvo** — o pecado mais traiçoeiro, e o que veremos em ação neste capítulo.

### Estratificação: Preservando a Realidade

O dataset é desbalanceado (63,3% Benigno, 36,7% Maligno). Uma amostragem aleatória simples poderia, por azar, criar um teste com proporções distorcidas. A **estratificação** garante que a proporção de classes seja preservada em treino e teste. Para dados desbalanceados, não é opcional.

O `oncoclassify_utils` já realiza essa separação estratificada uma única vez, com `random_state=42`, e a expõe pronta — de modo que `X_train`, `X_test`, `y_train`, `y_test` sejam **idênticos em todos os capítulos**. Basta importar e conferir.

#### Código 4.1: A Separação Sagrada (verificação)


In [ ]:
from oncoclassify_utils import X_train, X_test, y_train, y_test, y

print("--- Dimensoes ---")
print(f"Treino: {X_train.shape}   Teste: {X_test.shape}\n")

print("--- Proporcao de classes (0=Maligno, 1=Benigno) ---")
for nome, alvo in [("Completo", y), ("Treino", y_train), ("Teste", y_test)]:
    m, b = (alvo == 0).sum(), (alvo == 1).sum()
    print(f"{nome:9s} Maligno {m:>3} ({m/len(alvo)*100:5.2f}%)  "
          f"Benigno {b:>3} ({b/len(alvo)*100:5.2f}%)  Total {len(alvo)}")

--- Dimensoes ---
Treino: (480, 30)   Teste: (120, 30)

--- Proporcao de classes (0=Maligno, 1=Benigno) ---
Completo  Maligno 220 (36.67%)  Benigno 380 (63.33%)  Total 600
Treino    Maligno 176 (36.67%)  Benigno 304 (63.33%)  Total 480
Teste     Maligno  44 (36.67%)  Benigno  76 (63.33%)  Total 120


Temos **480 amostras** para desenvolver e **120** guardadas no cofre. A proporção é **idêntica** (36,67% / 63,33%) nos três conjuntos: a estratificação funcionou. A partir daqui, **todas** as análises usam apenas `X_train`/`y_train`.

#### Código 4.2: Verificação Visual da Estratificação


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from oncoclassify_utils import y, y_train, y_test, color_map, apply_hatches_bars

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (nome, alvo) in zip(axes, [("Completo", y), ("Treino", y_train), ("Teste", y_test)]):
    rotulos = alvo.map({0: "Maligno", 1: "Benigno"})
    sns.countplot(x=rotulos, order=["Benigno", "Maligno"], hue=rotulos,
                  palette=color_map, legend=False, ax=ax)
    apply_hatches_bars(ax, ["Benigno", "Maligno"])
    ax.set_title(f"{nome} (n={len(alvo)})"); ax.set_xlabel("Classe")

plt.suptitle("Distribuição de Classes: Completo vs Treino vs Teste", fontsize=14)
plt.tight_layout(); plt.show()

![Distribuição preservada nos três conjuntos](media/figura_04_1.png)

## O Pecado Original, Em Números

Agora, o cerne deste capítulo. No Capítulo 1 descartamos duas colunas "suspeitas" durante a curadoria: `diagnosis progression score` e `post evaluation flag`. Elas parecem preditores fabulosos. Vamos ver *quão* fabulosos — e por que isso é uma armadilha, não uma dádiva.

Comparamos três modelos de Regressão Logística por validação cruzada: (1) só as 30 *features* morfológicas legítimas; (2) as 30 mais as duas colunas suspeitas; (3) **apenas** as duas colunas suspeitas.

#### Código 4.3: Demonstração de Data Leakage


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from oncoclassify_utils import (df_raw, FEATURES_MORFOLOGICAS, COLUNAS_LEAKAGE)

# Partimos do bruto (sem duplicatas) so para EXPOR as colunas suspeitas.
df = df_raw.drop_duplicates().reset_index(drop=True)
alvo = df["target"].map({"M": 0, "B": 1})

def acc_cv(colunas):
    return cross_val_score(LogisticRegression(max_iter=5000),
                           df[colunas], alvo, cv=5, scoring="accuracy").mean()

print(f"So as 30 morfologicas (honesto):        {acc_cv(FEATURES_MORFOLOGICAS):.4f}")
print(f"Morfologicas + 2 colunas suspeitas:     {acc_cv(FEATURES_MORFOLOGICAS + COLUNAS_LEAKAGE):.4f}")
print(f"SO as 2 colunas suspeitas:              {acc_cv(COLUNAS_LEAKAGE):.4f}")

So as 30 morfologicas (honesto):        0.9533
Morfologicas + 2 colunas suspeitas:     1.0000
SO as 2 colunas suspeitas:              1.0000


> **⚠️ ATENÇÃO — Bom demais para ser verdade**
>
> As 30 medidas do núcleo celular — o sinal biológico real — chegam a **95,33%**. Ao adicionar as duas colunas suspeitas, a acurácia salta para **100%**. E, o veredicto: **apenas** essas duas colunas, sem nenhuma medida celular, também dão **100%**.
>
> Nenhum problema clínico real se resolve com 100% de acurácia. Quando isso acontece, não é genialidade — é **vazamento**. `diagnosis progression score` e `post evaluation flag` são *derivadas do próprio diagnóstico*: só existem *depois* que a verdade já é conhecida. Usá-las é perguntar ao paciente se ele tem câncer para então "prever" que ele tem. **Foi exatamente esse tipo de atalho que fez o OncoClassify 1.0 brilhar nos testes e falhar com Helena.** Por isso a curadoria as eliminou no Capítulo 1.

A lição é permanente: uma *feature* espetacularmente preditiva merece **desconfiança**, não celebração. Pergunte sempre: *essa informação estaria disponível no momento real da previsão?* Se a resposta for não, é vazamento.

## Validação Cruzada: O Teste Dentro do Treino

Se não podemos tocar no teste, como estimar a generalização durante o desenvolvimento? Com **Validação Cruzada (K-Fold)**: dividimos o **treino** em *K* partes (ex.: K=5); a cada rodada, treinamos em K−1 partes e validamos na parte restante; ao final, cada parte foi validação exatamente uma vez, e a performance é a média das K rodadas. Isso dá uma estimativa muito mais robusta do que uma única divisão de validação.

![Esquema do K-Fold (K=5)](media/figura_04_2.png)

Cada linha é um *fold*: a faixa vermelha (validação) desliza a cada rodada, cobrindo todo o conjunto. Como no *split* inicial, usamos a versão **estratificada** para preservar as proporções de classe em cada *fold*.

> **📌 CHECKLIST DO CAPÍTULO 4**
>
> ✅ Entende por que separar treino e teste é crítico — o teste simula dados futuros.
>
> ✅ Sabe o que é **Data Leakage** e viu, em números, duas colunas de vazamento darem **100%** de acurácia fraudulenta.
>
> ✅ Compreende a **estratificação** e confirmou que ela preservou 36,67%/63,33% nos três conjuntos.
>
> ✅ Confirmou os conjuntos canônicos: **X_train (480×30)** e **X_test (120×30)**.
>
> ✅ Entende o conceito de **Validação Cruzada** como avaliação robusta usando só o treino.
>
> **⚠️ CRÍTICO:** A partir daqui, esqueça que `X_test` existe. Toda análise, engenharia de *features*, seleção e ajuste acontece **exclusivamente** em `X_train`/`y_train`.

O comando foi executado. Em uma fração de segundo, o cofre foi trancado — e, mais importante, eu havia desmascarado o atalho que me traíra. Aquelas duas colunas sedutoras, com seus 100%, eram a cara do OncoClassify 1.0: uma certeza fabricada. Desta vez eu as reconheci e as descartei antes que pudessem fazer mal.

Com a fundação correta, eu podia finalmente mergulhar nos dados de treino sem medo de trapacear. Era hora de desenhar o mapa do terreno — a geografia das 30 *features* que descreviam a diferença entre a saúde e a doença.
